In [60]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [51]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 300)

In [79]:
df = pd.read_csv('run_4_27_2025/grad_frame_10_layer.csv')

In [80]:
df = df.drop(df.loc[df['token_id']==0].index)

In [81]:
df['token_id']=df['token_id'].astype(str)

In [82]:
df.head(20)

,orig_index,token_id,attribution_score,label,prediction_prob
0,20453,1,-0.304067,0.0,0.084188
1,20453,24,-6.350651,0.0,0.084188
2,20453,2,-1.930850,0.0,0.084188
3,20453,25,0.176169,0.0,0.084188
4,20453,4,-0.272070,0.0,0.084188
5,20453,5,-0.133688,0.0,0.084188
6,20453,63,-0.099698,0.0,0.084188
7,20453,78,0.302127,0.0,0.084188
8,20453,27,4.058901,0.0,0.084188
9,20453,36,-0.017207,0.0,0.084188


In [83]:
df['label'].value_counts()

label
0.0    1124916
1.0      79857
Name: count, dtype: int64

In [84]:
with open('vocab/id2token.json', 'r') as file:
    id2token = json.load(file)

with open('vocab/token2id.json', 'r') as file:
    token2id = json.load(file)

In [85]:
def encode(seq, *, mapping=token2id, unk_token="<UNK>"):
    """Convert list[str] -> list[int]."""
    unk = mapping[unk_token]
    return [mapping.get(tok, unk) for tok in seq]

def decode(id_seq, *, reverse_mapping=id2token):
    """Convert list[int] -> list[str]."""
    return [reverse_mapping[i] for i in id_seq]

In [86]:
df['token_id'] = decode(df['token_id'])

In [7]:
df_ch_only = df.loc[df['label']==1]

In [8]:
df_ch_only

,orig_index,token_id,attribution_score,label,prediction_prob
1040,128052,24,-1.145429,1.0,0.733358
1041,128052,24,-9.655292,1.0,0.733358
1042,128052,2,-3.252165,1.0,0.733358
1043,128052,4,-0.232236,1.0,0.733358
1044,128052,36,-2.106662,1.0,0.733358
...,...,...,...,...,...
2578280,127356,0,0.000000,1.0,0.161055
2578281,127356,0,0.000000,1.0,0.161055
2578282,127356,0,0.000000,1.0,0.161055
2578283,127356,0,0.000000,1.0,0.161055


In [40]:
pred = df_ch_only['prediction_prob'].quantile(0.7)

In [41]:
sort_95 = df_ch_only.loc[df_ch_only['prediction_prob']>pred].sort_values('prediction_prob', ascending=False)

In [42]:
top_25 = sort_95[['orig_index','label','prediction_prob']].drop_duplicates()[:30]['orig_index']

In [43]:
sort_95[['orig_index','label','prediction_prob']].drop_duplicates()[:30]

,orig_index,label,prediction_prob
229477,792,1.0,0.991545
336150,85396,1.0,0.991532
1991452,127654,1.0,0.991417
769460,42526,1.0,0.991396
1819351,169558,1.0,0.991395
934811,86584,1.0,0.991378
1340916,44282,1.0,0.991378
613146,84985,1.0,0.991378
772758,86013,1.0,0.991377
2436894,128315,1.0,0.991373


# Discards

In [11]:
mahjong_tiles = [
    # Manzu (Characters)
    "1m", "2m", "3m", "4m", "5m", "6m", "7m", "8m", "9m",
    
    # Souzu (Bamboos)
    "1s", "2s", "3s", "4s", "5s", "6s", "7s", "8s", "9s",
    
    # Pinzu (Circles)
    "1p", "2p", "3p", "4p", "5p", "6p", "7p", "8p", "9p",
    
    # Winds
    "East", "South", "West", "North",
    
    # Dragons (Colors)
    "White", "Green", "Red"
]

In [12]:
wind_translate = {0:"East",1:"South",2:"West",3:"North"}
is_riichi_translate={0:"No",1:"Yes"}

In [14]:
def display_discards(discard_df):
    print(f'Hand number: {discard_df['orig_index'][0]}')

    wind = discard_df['token_id'][0] - 34
    
    print(f'Wind: {wind}, {round(discard_df['attribution_score'][0],4)}')

    seat = discard_df['token_id'][1] - 38
    
    print(f'Seat: {seat}, {round(discard_df['attribution_score'][1],4)}')
    
    is_riichi = discard_df['token_id'][2] - 44
    
    print(f'Is riichi: {is_riichi_translate[is_riichi]}, {round(discard_df['attribution_score'][2],4)}')

    dora_ind = discard_df['token_id'][3] - 46
    
    print(f'Dora Indicator: {mahjong_tiles[dora_ind]}, {round(discard_df['attribution_score'][3],4)}')
    
    for i, r in discard_df[4:].iterrows():
        print(f'{mahjong_tiles[int(r['token_id'])]}, {round(r['attribution_score'],4)}')
    print('---------------------------------------------')

In [15]:
def display_discards_no_grad(discard_df):
    print(f'Hand number: {discard_df['orig_index'][0]}')

    wind = discard_df['token_id'][0] - 34
    
    print(f'Wind: {wind}')

    seat = discard_df['token_id'][1] - 38
    
    print(f'Seat: {seat}')
    
    is_riichi = discard_df['token_id'][2] - 44
    
    print(f'Is riichi: {is_riichi_translate[is_riichi]}')

    dora_ind = discard_df['token_id'][3] - 46
    
    print(f'Dora Indicator: {mahjong_tiles[dora_ind]}')
    
    for i, r in discard_df[4:].iterrows():
        print(f'{mahjong_tiles[int(r['token_id'])]}')
    print('---------------------------------------------')

In [ ]:
for hand in top_25:
    display_discards(df.loc[df['orig_index'] == hand].copy().reset_index())

In [ ]:
for hand in top_25:
    display(df.loc[df['orig_index'] == hand].copy().reset_index())

# Updated Grads

In [87]:
df.loc[df['orig_index'] == 792]

,orig_index,token_id,attribution_score,label,prediction_prob
229452,792,S,-0.019763,1.0,0.991545
229453,792,S,-0.608080,1.0,0.991545
229454,792,WU20,-0.014820,1.0,0.991545
229455,792,1st,0.006444,1.0,0.991545
229456,792,no_riichi,-0.040281,1.0,0.991545
229457,792,0_OR,-0.014579,1.0,0.991545
229458,792,LNorth,0.188170,1.0,0.991545
229459,792,PNorth,0.453856,1.0,0.991545
229460,792,LSouth,0.072493,1.0,0.991545
229461,792,RRed,0.145388,1.0,0.991545


# Greater Patterns

In [88]:
df

,orig_index,token_id,attribution_score,label,prediction_prob
0,20453,S,-0.304067,0.0,0.084188
1,20453,E,-6.350651,0.0,0.084188
2,20453,WOoA20,-1.930850,0.0,0.084188
3,20453,1st,0.176169,0.0,0.084188
4,20453,no_riichi,-0.272070,0.0,0.084188
...,...,...,...,...,...
2579652,48765,L1m,0.311494,0.0,0.003410
2579653,48765,P2m,0.173270,0.0,0.003410
2579654,48765,P1p,-0.464339,0.0,0.003410
2579655,48765,A8m,0.059956,0.0,0.003410


In [89]:
df[['token_id','attribution_score']].groupby('token_id').mean().reset_index().sort_values('attribution_score', ascending=False)

,token_id,attribution_score
49,P4p,0.509300
37,LEast,0.311645
27,L1m,0.284201
51,P6m,0.277938
19,ASouth,0.222464
21,AWhite,0.205184
38,LGreen,0.198467
53,P7p,0.188791
74,RWhite,0.174534
40,LRed,0.159660


In [90]:
len(df['token_id'].unique())

79

In [63]:
plt.hist(df[['token_id','attribution_score']].groupby('token_id').get_group('P4s')['attribution_score'], bins=100)

KeyError: 'P4s'